# 1. Tổng quan dữ liệu

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Tự tìm dataset từ thư mục notebook
current = Path.cwd().resolve()

DATA_PATH = None
SEARCH_ROOTS = [Path.cwd().resolve(), *Path.cwd().resolve().parents]

for folder in SEARCH_ROOTS:
    candidates = [
        folder / "data" / "raw" / "RT_IOT2022.csv",
        folder / "result" / "preprocessing" / "raw data" / "RT_IOT2022.csv",
        folder / "final_IoT" / "sample_project" / "result" / "preprocessing" / "raw data" / "RT_IOT2022.csv",
    ]
    for candidate in candidates:
        if candidate.is_file():
            DATA_PATH = candidate
            break
    if DATA_PATH is not None:
        break

if DATA_PATH is None:
    raise FileNotFoundError("Khong tim thay RT_IOT2022.csv trong cau truc project")

df = pd.read_csv(DATA_PATH)

print("Kích thước dữ liệu:", df.shape)
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])

display(df.head())

### Nhận xét: Tổng quan dữ liệu

- Bộ dữ liệu RT_IOT2022 gồm **123.117 bản ghi và 85 thuộc tính**.
- Các thuộc tính mô tả những đặc điểm liên quan đến lưu lượng mạng IoT.
- Dữ liệu bao gồm các thông tin về giao thức mạng, dịch vụ, thời gian và thống kê gói tin.

**Kết luận:** Bộ dữ liệu có quy mô tương đối lớn và nhiều thuộc tính, cần tiếp tục phân tích chất lượng dữ liệu, phân bố đặc trưng và phân bố nhãn trước khi xây dựng mô hình.

# 2. Kiểm tra cấu trúc dữ liệu

In [ ]:
print("=== THÔNG TIN DATASET ===")
df.info()

print("\n=== THỐNG KÊ KIỂU DỮ LIỆU ===")
print(df.dtypes.value_counts())

print("\n=== DANH SÁCH TẤT CẢ THUỘC TÍNH ===")

columns_info = pd.DataFrame({
    "STT": range(1, len(df.columns) + 1),
    "Tên thuộc tính": df.columns,
    "Kiểu dữ liệu": df.dtypes.astype(str).values,
    "Số giá trị khác nhau": df.nunique().values
})

display(columns_info)

### Nhận xét: Kiểu dữ liệu và thuộc tính

- Bộ dữ liệu có **82 thuộc tính dạng số** và **3 thuộc tính dạng object**.
- Các thuộc tính dạng object gồm `proto`, `service` và `Attack_type`.
- `Attack_type` là nhãn mục tiêu của bài toán phân loại.
- Phát hiện thuộc tính `bwd_URG_flag_count` chỉ chứa một giá trị duy nhất.
- Cột `Unnamed: 0` có khả năng là chỉ số được tạo khi xuất dữ liệu ra CSV.

**Kết luận:** Cần xem xét mã hóa các đặc trưng phân loại `proto` và `service`, đồng thời xem xét loại bỏ cột hằng và cột chỉ số trong giai đoạn tiền xử lý.

# 3. Kiểm tra chất lượng của dữ liệu

In [ ]:
print("=== GIÁ TRỊ THIẾU ===")
print("Tổng giá trị thiếu:", df.isnull().sum().sum())

print("\n=== GIÁ TRỊ VÔ HẠN ===")

numeric_df = df.select_dtypes(include=np.number)

print("Tổng giá trị Inf:",
      np.isinf(numeric_df.to_numpy()).sum())

print("\n=== DỮ LIỆU TRÙNG LẶP ===")

# Kiểm tra toàn bộ các cột
print("Số dòng trùng toàn bộ:",
      df.duplicated().sum())

# Không sử dụng cột chỉ số trong phép so sánh
feature_cols = df.columns.drop("Unnamed: 0", errors="ignore")

print("Số dòng trùng khi bỏ Unnamed: 0:",
      df.duplicated(subset=feature_cols).sum())

print("\n=== CÁC CỘT CHỈ CÓ MỘT GIÁ TRỊ ===")

constant_cols = df.columns[df.nunique(dropna=False) == 1]

print(constant_cols.tolist())

### Nhận xét: Chất lượng dữ liệu

- **Giá trị thiếu:** Không phát hiện giá trị thiếu.
- **Giá trị vô hạn:** Không phát hiện giá trị Inf.
- **Dữ liệu trùng lặp:** Khi xét toàn bộ 85 thuộc tính, không phát hiện dòng trùng hoàn toàn.
- Khi không xét cột `Unnamed: 0`, phát hiện **5.195 dòng trùng lặp**.
- **Thuộc tính hằng:** Phát hiện `bwd_URG_flag_count` chỉ có một giá trị duy nhất.

**Kết luận:**

- Bộ dữ liệu không cần xử lý giá trị thiếu hoặc vô hạn ở thời điểm khảo sát.
- Cần xem xét loại bỏ các bản ghi trùng lặp trong giai đoạn Data Cleaning.
- Cần xem xét loại bỏ các thuộc tính không mang thông tin phân biệt.
- Chưa thực hiện thay đổi dữ liệu trong giai đoạn EDA.

# 4. Thống kê mô tả các đặc trưng

In [ ]:
# Không đưa cột chỉ số vào phân tích
numeric_df = df.select_dtypes(include=np.number).drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

# Tính thống kê mô tả
stats = numeric_df.describe().T

stats = stats.rename(columns={
    "count": "Count",
    "mean": "Mean",
    "std": "Std",
    "min": "Min",
    "25%": "Q1",
    "50%": "Median",
    "75%": "Q3",
    "max": "Max"
})

# Bổ sung chỉ số để phát hiện vấn đề
stats["Skewness"] = numeric_df.skew()
stats["Zero_Percent"] = (
    numeric_df.eq(0).mean() * 100
)
stats["Unique_Values"] = numeric_df.nunique()

print("=== THỐNG KÊ MÔ TẢ ===")
display(stats.round(3))

# 1. Các thuộc tính có phân phối lệch mạnh
skewed = stats[
    stats["Skewness"].abs() > 3
].sort_values("Skewness", key=lambda x: x.abs(), ascending=False)

print("\n=== THUỘC TÍNH CÓ |SKEWNESS| > 3 ===")
print("Số lượng:", len(skewed))
display(
    skewed[
        ["Mean", "Median", "Std", "Skewness"]
    ].head(15).round(3)
)

# 2. Các thuộc tính chứa nhiều giá trị 0
zero_heavy = stats[
    stats["Zero_Percent"] >= 95
].sort_values("Zero_Percent", ascending=False)

print("\n=== THUỘC TÍNH CÓ >= 95% GIÁ TRỊ BẰNG 0 ===")
print("Số lượng:", len(zero_heavy))
display(
    zero_heavy[
        ["Min", "Median", "Max", "Zero_Percent"]
    ].round(3)
)

# 3. Các thuộc tính chỉ có một giá trị
constant = stats[
    stats["Unique_Values"] == 1
]

print("\n=== THUỘC TÍNH HẰNG ===")
display(constant[["Min", "Max", "Unique_Values"]])

### Nhận xét: Thống kê mô tả

**1. Phân phối dữ liệu**

- Phát hiện 65 thuộc tính số có độ lệch tuyệt đối
  (|Skewness|) lớn hơn 3.
- Một số thuộc tính như `bwd_bulk_bytes`,
  `bwd_bulk_packets`, `active.std` có độ lệch phải
  rất lớn.
- Điều này cho thấy nhiều đặc trưng không có
  phân phối đối xứng, phần lớn giá trị tập trung
  ở vùng thấp và có một số giá trị rất lớn.

**2. Đặc trưng chứa nhiều giá trị bằng 0**

- Phát hiện 16 thuộc tính có ít nhất 95%
  giá trị bằng 0.
- `flow_CWR_flag_count` có khoảng 99,94%
  giá trị bằng 0.
- `fwd_bulk_bytes` có khoảng 99,80%
  giá trị bằng 0.
- `bwd_URG_flag_count` có 100% giá trị bằng 0.

**3. Thang đo dữ liệu**

- Các đặc trưng có phạm vi giá trị khác nhau
  đáng kể.
- Ví dụ, `flow_duration` có giá trị lớn nhất
  khoảng 21.728, trong khi `flow_iat.tot`
  có giá trị lớn nhất trên 21 tỷ.
- Sự khác biệt thang đo có thể ảnh hưởng
  đến quá trình học của MLP.

**Kết luận:** Dataset có nhiều đặc trưng phân phối
lệch và thang đo không đồng nhất. Đây là những
vấn đề cần được phân tích sâu hơn trước khi
xây dựng pipeline tiền xử lý.

# 5. Phân tích phân bố nhãn

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

target = "Attack_type"

class_counts = df[target].value_counts()

class_distribution = pd.DataFrame({
    "Số mẫu": class_counts,
    "Tỷ lệ (%)": (
        class_counts / len(df) * 100
    ).round(3)
})

print("=== PHÂN BỐ NHÃN ===")
display(class_distribution)

print("\nSố lớp:", df[target].nunique())
print("Lớp nhiều nhất:", class_counts.idxmax())
print("Lớp ít nhất:", class_counts.idxmin())

imbalance_ratio = (
    class_counts.max() / class_counts.min()
)

print(f"Tỷ lệ mất cân bằng: {imbalance_ratio:.2f}:1")

# Vẽ biểu đồ
plt.figure(figsize=(12, 6))

sns.barplot(
    x=class_counts.values,
    y=class_counts.index,
    color="steelblue"
)

plt.title("Phân bố số lượng mẫu theo Attack_type")
plt.xlabel("Số lượng mẫu")
plt.ylabel("Loại tấn công")

plt.tight_layout()
plt.show()

### Nhận xét: Phân bố nhãn mục tiêu

- Dataset gồm 12 lớp thuộc tính `Attack_type`.
- Phân bố số lượng mẫu giữa các lớp không đồng đều.
- Lớp `DOS_SYN_Hping` chiếm phần lớn dữ liệu,
  với 94.659 mẫu, tương đương khoảng 76,89%.
- Lớp `NMAP_FIN_SCAN` chỉ có 28 mẫu.
- Lớp `Metasploit_Brute_Force_SSH` chỉ có 37 mẫu.
- Tỷ lệ giữa lớp nhiều mẫu nhất và ít mẫu nhất
  khoảng 3.380,68:1.

**Kết luận:** Mất cân bằng lớp là một vấn đề
quan trọng cần xem xét trong quá trình xây dựng
và đánh giá mô hình MLP.

# 6. Các vấn đề được phát hiện từ dataset

In [ ]:
import pandas as pd
import numpy as np

target = "Attack_type"

# Chỉ lấy đặc trưng số, không tính cột chỉ số
numeric_features = df.select_dtypes(
    include=np.number
).drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

# 1. Chất lượng dữ liệu
missing_count = int(df.isna().sum().sum())

inf_count = int(
    np.isinf(numeric_features.to_numpy()).sum()
)

duplicate_count = int(
    df.duplicated(
        subset=df.columns.drop(
            "Unnamed: 0",
            errors="ignore"
        )
    ).sum()
)

# 2. Thuộc tính hằng
constant_cols = df.columns[
    df.nunique(dropna=False) == 1
].tolist()

# 3. Phân phối lệch
skewness = numeric_features.skew()

skewed_cols = skewness[
    skewness.abs() > 3
].index.tolist()

# 4. Nhiều giá trị bằng 0
zero_percentage = (
    numeric_features.eq(0).mean() * 100
)

high_zero_cols = zero_percentage[
    zero_percentage >= 95
].index.tolist()

# 5. Mất cân bằng nhãn
class_counts = df[target].value_counts()

imbalance_ratio = (
    class_counts.max() / class_counts.min()
)

# Tổng hợp vào bảng
issues = pd.DataFrame({
    "Vấn đề": [
        "Giá trị thiếu",
        "Giá trị vô hạn",
        "Dữ liệu trùng lặp",
        "Thuộc tính hằng",
        "Phân phối lệch mạnh |Skew| > 3",
        "Thuộc tính có >= 95% giá trị 0",
        "Mất cân bằng nhãn"
    ],
    "Kết quả": [
        missing_count,
        inf_count,
        duplicate_count,
        len(constant_cols),
        len(skewed_cols),
        len(high_zero_cols),
        round(imbalance_ratio, 2)
    ],
    "Hướng nghiên cứu": [
        "Xem xét xử lý nếu có",
        "Xem xét xử lý nếu có",
        "Kiểm tra trùng lặp trước khi chia dữ liệu",
        "Đánh giá loại bỏ đặc trưng hằng",
        "Phân tích Histogram, KDE, Boxplot",
        "Đánh giá độ hữu ích của đặc trưng",
        "Phân tích lớp hiếm và phương pháp cân bằng"
    ]
})

print("=== TỔNG HỢP VẤN ĐỀ DATASET ===")
display(issues)

print("\n=== DANH SÁCH CỘT HẰNG ===")
print(constant_cols)

print("\n=== CÁC THUỘC TÍNH LỆCH MẠNH NHẤT ===")
display(
    skewness.abs()
    .sort_values(ascending=False)
    .head(10)
    .to_frame("Absolute Skewness")
)

print("\n=== CÁC THUỘC TÍNH NHIỀU GIÁ TRỊ 0 NHẤT ===")
display(
    zero_percentage
    .sort_values(ascending=False)
    .head(10)
    .to_frame("Zero (%)")
)